In [ ]:
import pandas as pd
import plotly.express as px
import os
from pathlib import Path
import h3

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'configs/salish_sea.yaml').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the Viewshed Toolkit repository root.')


REPO_ROOT = find_repo_root()
H3_RESOLUTION = 7
SIGHTINGS_PATH = REPO_ROOT / 'data/processed/domain/whale_layer/sightings/imputed_retrospective.parquet'
LAND_VIEWSHED_PATH = REPO_ROOT / f'data/processed/domain/human/viewshed/RES{H3_RESOLUTION}/LAND_STATIC_WEIGHTS_R{H3_RESOLUTION}.parquet'
WATER_VIEWSHED_PATH = REPO_ROOT / f'data/processed/domain/human/viewshed/RES{H3_RESOLUTION}/WATER_STATIC_WEIGHTS_R{H3_RESOLUTION}.parquet'

In [ ]:
if os.path.exists(SIGHTINGS_PATH):
    sightings = pd.read_parquet(SIGHTINGS_PATH)
else:
    raise FileNotFoundError(f"Sightings not created: {SIGHTINGS_PATH}")

if os.path.exists(LAND_VIEWSHED_PATH):
    viewshed_land = pd.read_parquet(LAND_VIEWSHED_PATH)
else:
    raise FileNotFoundError(f"Land viewshed not created: {LAND_VIEWSHED_PATH}")

if os.path.exists(WATER_VIEWSHED_PATH):
    viewshed_water = pd.read_parquet(WATER_VIEWSHED_PATH)
else:
    raise FileNotFoundError(f"Water viewshed not created: {WATER_VIEWSHED_PATH}")

In [ ]:
net_unique_cells = (
    set(viewshed_water["target_h3"].unique())
    | set(viewshed_land["target_h3"].unique())
)

print(len(net_unique_cells))

## Build a common target-cell analysis table

The unit of analysis is one modeled target H3 cell. Sightings outside the viewshed target universe are excluded, while modeled cells with no sightings are retained as zeroes. `sighting_days` counts distinct cell-days so duplicate reports on the same day do not inflate the response.

In [ ]:
h3_col = f"RES{H3_RESOLUTION}"

sightings_cells = sightings[
    ["SIGHTING_DATE_UTC", "LATITUDE", "LONGITUDE", "SOURCE", "ECOTYPE_DETAIL_EFFECTIVE"]
].rename(
    columns={"SIGHTING_DATE_UTC": "DATETIME", "ECOTYPE_DETAIL_EFFECTIVE": "TYPE"}
).copy()
sightings_cells["DATETIME"] = pd.to_datetime(sightings_cells["DATETIME"])
sightings_cells = sightings_cells.dropna(
    subset=["DATETIME", "LATITUDE", "LONGITUDE"]
)
sightings_cells["DATE"] = sightings_cells["DATETIME"].dt.normalize()
sightings_cells["YEAR"] = sightings_cells["DATETIME"].dt.year
sightings_cells["HALF_DECADE"] = (sightings_cells["YEAR"] // 5) * 5
sightings_cells[h3_col] = [
    h3.latlng_to_cell(lat, lon, H3_RESOLUTION)
    for lat, lon in zip(sightings_cells["LATITUDE"], sightings_cells["LONGITUDE"])
]

viewshed_sightings = sightings_cells[
    sightings_cells[h3_col].isin(net_unique_cells)
].copy()

print(f"Sightings in target universe: {len(viewshed_sightings):,} / {len(sightings_cells):,}")
print(f"Unique sighting cells: {viewshed_sightings[h3_col].nunique():,}")

In [ ]:
def summarize_viewshed(frame, prefix):
    working = frame[["source_h3", "target_h3", "weight_static_viewability"]].copy()
    working["weight_static_viewability"] = (
        working["weight_static_viewability"].fillna(0.0)
    )
    summary = (
        working.groupby("target_h3", as_index=False)
        .agg(
            **{
                f"{prefix}_viewability_sum": ("weight_static_viewability", "sum"),
                f"{prefix}_candidate_sources": ("source_h3", "nunique"),
                f"{prefix}_candidate_mean_viewability": ("weight_static_viewability", "mean"),
                f"{prefix}_max_source_viewability": ("weight_static_viewability", "max"),
            }
        )
    )
    positive = (
        working[working["weight_static_viewability"] > 0]
        .groupby("target_h3", as_index=False)
        .agg(**{f"{prefix}_visible_sources": ("source_h3", "nunique")})
    )
    return summary.merge(positive, on="target_h3", how="left").rename(
        columns={"target_h3": h3_col}
    )


land_features = summarize_viewshed(viewshed_land, "land")
water_features = summarize_viewshed(viewshed_water, "water")

In [ ]:
analysis_df = pd.DataFrame({h3_col: sorted(net_unique_cells)})
analysis_df = analysis_df.merge(land_features, on=h3_col, how="left")
analysis_df = analysis_df.merge(water_features, on=h3_col, how="left")

viewability_columns = [
    "land_viewability_sum",
    "land_candidate_sources",
    "land_candidate_mean_viewability",
    "land_visible_sources",
    "land_max_source_viewability",
    "water_viewability_sum",
    "water_candidate_sources",
    "water_candidate_mean_viewability",
    "water_visible_sources",
    "water_max_source_viewability",
]
analysis_df[viewability_columns] = analysis_df[viewability_columns].fillna(0)
analysis_df["total_viewability_sum"] = (
    analysis_df["land_viewability_sum"] + analysis_df["water_viewability_sum"]
)
analysis_df["total_visible_sources"] = (
    analysis_df["land_visible_sources"] + analysis_df["water_visible_sources"]
).astype(int)
analysis_df["total_candidate_sources"] = (
    analysis_df["land_candidate_sources"] + analysis_df["water_candidate_sources"]
).astype(int)
analysis_df["all_pair_mean_viewability"] = analysis_df[
    "total_viewability_sum"
].div(analysis_df["total_candidate_sources"].where(analysis_df["total_candidate_sources"] > 0))
analysis_df["balanced_domain_mean_viewability"] = analysis_df[
    ["land_candidate_mean_viewability", "water_candidate_mean_viewability"]
].mean(axis=1)
analysis_df["land_mean_percentile"] = analysis_df[
    "land_candidate_mean_viewability"
].rank(pct=True)
analysis_df["water_mean_percentile"] = analysis_df[
    "water_candidate_mean_viewability"
].rank(pct=True)
analysis_df["balanced_domain_percentile"] = analysis_df[
    ["land_mean_percentile", "water_mean_percentile"]
].mean(axis=1)
analysis_df["max_source_viewability"] = analysis_df[
    ["land_max_source_viewability", "water_max_source_viewability"]
].max(axis=1)
analysis_df["mean_viewability_per_visible_source"] = analysis_df[
    "total_viewability_sum"
].div(analysis_df["total_visible_sources"].where(analysis_df["total_visible_sources"] > 0))

In [ ]:
sighting_features = (
    viewshed_sightings.groupby(h3_col, as_index=False)
    .agg(
        sighting_days=("DATE", "nunique"),
        sighting_records=("DATE", "size"),
        first_sighting=("DATE", "min"),
        last_sighting=("DATE", "max"),
    )
)
analysis_df = analysis_df.merge(sighting_features, on=h3_col, how="left")
analysis_df[["sighting_days", "sighting_records"]] = analysis_df[
    ["sighting_days", "sighting_records"]
].fillna(0).astype(int)
analysis_df["ever_sighted"] = (analysis_df["sighting_days"] > 0).astype(int)

display(analysis_df.head())
display(analysis_df.describe(include="all").T)

## Overall associations

Pearson measures linear association; Spearman measures monotonic association and is less dominated by the long right tails. Correlation with `ever_sighted` is the point-biserial correlation because that field is binary.

In [ ]:
metrics = [
    "total_viewability_sum",
    "total_visible_sources",
    "sighting_days",
    "ever_sighted",
]

print("Pearson")
display(analysis_df[metrics].corr(method="pearson"))

print("Spearman")
display(analysis_df[metrics].corr(method="spearman"))

extended_metrics = [
    "land_viewability_sum",
    "land_candidate_mean_viewability",
    "water_viewability_sum",
    "water_candidate_mean_viewability",
    "all_pair_mean_viewability",
    "balanced_domain_mean_viewability",
    "balanced_domain_percentile",
    "mean_viewability_per_visible_source",
    "max_source_viewability",
    "sighting_days",
    "sighting_records",
    "ever_sighted",
]
print("Extended Spearman diagnostics")
display(analysis_df[extended_metrics].corr(method="spearman"))

## Binned relationship

Equal-count viewability bins make nonlinear trends visible. The ever-sighted rate is the cleanest first diagnostic; mean and median sighting days show whether repeated sightings rise as well.

In [ ]:
analysis_df["viewability_decile"] = pd.qcut(
    analysis_df["total_viewability_sum"].rank(method="first"),
    q=10,
    labels=False,
)

binned = (
    analysis_df.groupby("viewability_decile", as_index=False)
    .agg(
        median_viewability=("total_viewability_sum", "median"),
        mean_sighting_days=("sighting_days", "mean"),
        median_sighting_days=("sighting_days", "median"),
        proportion_ever_sighted=("ever_sighted", "mean"),
        mean_visible_sources=("total_visible_sources", "mean"),
        cells=(h3_col, "size"),
    )
)

display(binned)

In [ ]:
plot_binned = binned[binned["median_viewability"] > 0]
fig = px.line(
    plot_binned,
    x="median_viewability",
    y="proportion_ever_sighted",
    markers=True,
    log_x=True,
    title="Probability a cell was ever sighted by viewshed decile",
)
fig.show()

fig = px.line(
    plot_binned,
    x="median_viewability",
    y="mean_sighting_days",
    markers=True,
    log_x=True,
    title="Mean distinct sighting days by viewshed decile",
)
fig.show()

## Ranking diagnostics by observability component

If the viewshed pinpoints high-sighting cells, higher score deciles should have a higher ever-sighted rate and capture a disproportionate share of sighting days. Land and water components are evaluated separately because their spatial meanings differ. The comparison includes the raw land-plus-water sum, a mean across every candidate source pair, an equal-weight mean of the land and water candidate-pair means, and an equal-weight average of their percentile ranks. These normalized scores are diagnostics; they should only replace the raw score if they improve the relationship and match the intended observer-effort interpretation.

In [ ]:
ranking_rows = []
ranking_metrics = {
    "raw total viewability": "total_viewability_sum",
    "candidate-pair mean": "all_pair_mean_viewability",
    "equal-domain mean": "balanced_domain_mean_viewability",
    "equal-domain percentile": "balanced_domain_percentile",
    "land viewability sum": "land_viewability_sum",
    "land candidate mean": "land_candidate_mean_viewability",
    "water viewability sum": "water_viewability_sum",
    "water candidate mean": "water_candidate_mean_viewability",
    "visible sources": "total_visible_sources",
}

for label, score_column in ranking_metrics.items():
    ranked = analysis_df[[h3_col, score_column, "sighting_days", "ever_sighted"]].copy()
    ranked["score_decile"] = pd.qcut(
        ranked[score_column].rank(method="first"),
        q=10,
        labels=False,
    )
    summary = (
        ranked.groupby("score_decile", as_index=False)
        .agg(
            median_score=(score_column, "median"),
            proportion_ever_sighted=("ever_sighted", "mean"),
            sighting_days=("sighting_days", "sum"),
            cells=(h3_col, "size"),
        )
    )
    summary["score"] = label
    summary["sighting_day_share"] = (
        summary["sighting_days"] / analysis_df["sighting_days"].sum()
    )
    ranking_rows.append(summary)

ranking_by_decile = pd.concat(ranking_rows, ignore_index=True)
baseline_rate = analysis_df["ever_sighted"].mean()
ranking_by_decile["ever_sighted_lift"] = (
    ranking_by_decile["proportion_ever_sighted"] / baseline_rate
)
display(ranking_by_decile)

top_decile_summary = ranking_by_decile[ranking_by_decile["score_decile"] == 9][
    ["score", "proportion_ever_sighted", "ever_sighted_lift", "sighting_day_share"]
].sort_values("ever_sighted_lift", ascending=False)
display(top_decile_summary)

In [ ]:
fig = px.line(
    ranking_by_decile,
    x="score_decile",
    y="proportion_ever_sighted",
    color="score",
    markers=True,
    title="Ever-sighted rate across observability score deciles",
)
fig.add_hline(
    y=baseline_rate,
    line_dash="dot",
    line_color="gray",
    annotation_text="all-cell baseline",
)
fig.show()

## Correlation by five-year period

Each period uses the complete target-cell universe and fills unsighted cells with zero. The final period may contain fewer than five observed years, so `observed_years` and the actual label are included.

In [ ]:
half_decade_rows = []
view_metrics = [
    "total_viewability_sum",
    "total_visible_sources",
    "land_viewability_sum",
    "water_viewability_sum",
    "land_candidate_mean_viewability",
    "water_candidate_mean_viewability",
    "all_pair_mean_viewability",
    "balanced_domain_mean_viewability",
    "balanced_domain_percentile",
    "mean_viewability_per_visible_source",
    "max_source_viewability",
]

for period_start, period_sightings in viewshed_sightings.groupby("HALF_DECADE"):
    period_counts = period_sightings.groupby(h3_col)["DATE"].nunique()
    period_df = analysis_df[[h3_col] + view_metrics].copy()
    period_df["sighting_days"] = (
        period_df[h3_col].map(period_counts).fillna(0).astype(int)
    )
    period_df["ever_sighted"] = (period_df["sighting_days"] > 0).astype(int)
    observed_years = sorted(period_sightings["YEAR"].unique())
    period_end = int(max(observed_years))

    for method in ["pearson", "spearman"]:
        corr = period_df[view_metrics + ["sighting_days", "ever_sighted"]].corr(
            method=method
        )
        row = {
            "period_start": int(period_start),
            "period": f"{int(period_start)}–{period_end}",
            "observed_years": len(observed_years),
            "method": method,
            "sighting_days": int(period_df["sighting_days"].sum()),
            "cells_ever_sighted": int(period_df["ever_sighted"].sum()),
        }
        for metric in view_metrics:
            row[f"{metric}_vs_sighting_days"] = corr.loc[metric, "sighting_days"]
            row[f"{metric}_vs_ever_sighted"] = corr.loc[metric, "ever_sighted"]
        half_decade_rows.append(row)

half_decade_correlations = pd.DataFrame(half_decade_rows)
display(half_decade_correlations)

In [ ]:
correlation_plot = half_decade_correlations.melt(
    id_vars=["period_start", "period", "method"],
    value_vars=[
        "total_viewability_sum_vs_sighting_days",
        "total_viewability_sum_vs_ever_sighted",
        "total_visible_sources_vs_sighting_days",
        "total_visible_sources_vs_ever_sighted",
        "land_viewability_sum_vs_ever_sighted",
        "water_viewability_sum_vs_ever_sighted",
    ],
    var_name="comparison",
    value_name="correlation",
)
fig = px.line(
    correlation_plot,
    x="period_start",
    y="correlation",
    color="comparison",
    facet_row="method",
    markers=True,
    title="Viewshed–sightings association by five-year period",
)
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.show()

## Effect-size summaries

These are easier to interpret than a correlation alone: compare sighted and unsighted cells, and measure top-decile lift relative to the bottom decile.

In [ ]:
sighted_comparison = (
    analysis_df.groupby("ever_sighted", as_index=False)
    .agg(
        cells=(h3_col, "size"),
        median_total_viewability=("total_viewability_sum", "median"),
        mean_total_viewability=("total_viewability_sum", "mean"),
        median_visible_sources=("total_visible_sources", "median"),
        mean_visible_sources=("total_visible_sources", "mean"),
    )
)
display(sighted_comparison)

bottom_rate = binned.loc[
    binned["viewability_decile"] == binned["viewability_decile"].min(),
    "proportion_ever_sighted",
].iloc[0]
top_rate = binned.loc[
    binned["viewability_decile"] == binned["viewability_decile"].max(),
    "proportion_ever_sighted",
].iloc[0]
top_decile_lift = top_rate / bottom_rate if bottom_rate > 0 else float("inf")
print(f"Bottom-decile ever-sighted rate: {bottom_rate:.1%}")
print(f"Top-decile ever-sighted rate: {top_rate:.1%}")
print(f"Top-vs-bottom decile lift: {top_decile_lift:.2f}x")

## Interpretation boundaries

A positive relationship supports the claim that the viewshed identifies locations where sightings are more likely to be recorded. It does **not** establish that whales prefer highly observable cells. Sightings reflect whale presence, reporting effort, vessel/shore access, source coverage, and spatial autocorrelation. Raw land and water sums are opportunity totals and are affected by the number of candidate sources. Candidate-pair means and equal-domain scores test scale normalization, but equal weighting is not automatically scientifically valid. In particular, water-source observability needs an empirical vessel/observer-effort prior (for example AIS, ferry, or survey effort) before it should be combined operationally with land observability. The five-year results help test temporal stability, but a later spatial or count model should control for coastline/access covariates and reporting source, and should use spatially blocked uncertainty estimates rather than treating H3 cells as independent.